# 🛰️ ESG Ratings Are Self-Reported. Satellite Data Isn't.

This notebook cross-references the **NO₂ emission trend** at the primary production facilities of 6 European industrial companies against their **5-year stock return**, using:

- **[Jiskta API](https://jiskta.com)** — Copernicus CAMS reanalysis, `aggregate=trend` to get per-facility NO₂ slope (µg/m³/year)
- **[yfinance](https://pypi.org/project/yfinance/)** — total stock return 2018–2024

**Results from running this notebook (March 2026, Jiskta API live data):**

| Company | NO₂ slope (µg/m³/yr) | Return 2018–2024 | Notes |
|---------|----------------------|------------------|---------|
| RWE (Cologne) | −1.095 | +190.8% | Coal exit visible from orbit ✅ |
| ArcelorMittal (Ghent) | −1.033 | −11.9% | Improving facility, sector headwinds ⚠️ |
| BASF (Ludwigshafen) | −0.933 | −25.3% | Improving but gas-crisis exposed ⚠️ |
| Volkswagen (Wolfsburg) | −0.422 | −0.3% | Slight improvement, flat stock 🟡 |
| Thyssenkrupp (Duisburg) | −0.385 | −72.8% | Structural issues dominate ❌ |
| **HeidelbergMaterials (Leimen)** | **+0.930** | **+7.6%** | **WORSENING — green narrative divergence 🚩** |

**R² = 0.08 (near zero)** — markets are not pricing emission trajectories. The actionable signal is divergence, not correlation.

**Requirements**
```
pip install jiskta yfinance pandas matplotlib numpy
```

In [ ]:
# !pip install "git+https://TOKEN@github.com/fvsever/jiskta-python.git#egg=jiskta[pandas]" yfinance matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from jiskta import JisktaClient
import yfinance as yf

API_KEY = "sk_live_YOUR_KEY_HERE"  # get one at https://jiskta.com
client = JisktaClient(api_key=API_KEY)

print("SDK ready")

## Define companies and their primary facility locations

We choose 6 European heavy-industry companies. For each we define a **tight bounding box around their main production facility** — not the HQ. This is crucial: an office building in Paris has the same ambient air quality as its neighbours. A steel smelter in Ghent does not.

| Company | Facility | Why here |
|---------|----------|----------|
| ArcelorMittal | Ghent, Belgium | Largest EU blast furnace |
| Thyssenkrupp | Duisburg, Germany | Ruhr steel complex |
| BASF | Ludwigshafen, Germany | Largest chemical complex in the world |
| RWE | Cologne area, Germany | Power generation fleet |
| Volkswagen | Wolfsburg, Germany | Main assembly plant |
| HeidelbergMaterials | Leimen, Germany | Primary cement operations |

In [ ]:
companies = [
    {
        "name": "ArcelorMittal (Ghent)",
        "ticker": "MT",
        "lat": (51.0, 51.2),
        "lon": (3.6, 3.9),
        "sector": "Steel",
        "color": "#ef4444",
    },
    {
        "name": "Thyssenkrupp (Duisburg)",
        "ticker": "TKA.DE",
        "lat": (51.4, 51.6),
        "lon": (6.7, 6.9),
        "sector": "Steel",
        "color": "#ef4444",
    },
    {
        "name": "BASF (Ludwigshafen)",
        "ticker": "BAS.DE",
        "lat": (49.4, 49.6),
        "lon": (8.3, 8.5),
        "sector": "Chemicals",
        "color": "#f97316",
    },
    {
        "name": "RWE (Cologne area)",
        "ticker": "RWE.DE",
        "lat": (50.9, 51.1),
        "lon": (6.5, 6.8),
        "sector": "Power",
        "color": "#22c55e",
    },
    {
        "name": "Volkswagen (Wolfsburg)",
        "ticker": "VOW3.DE",
        "lat": (52.3, 52.5),
        "lon": (10.7, 10.9),
        "sector": "Automotive",
        "color": "#3b82f6",
    },
    {
        "name": "HeidelbergMaterials (Leimen)",
        "ticker": "HEI.DE",
        "lat": (49.3, 49.4),
        "lon": (8.6, 8.7),
        "sector": "Cement",
        "color": "#a855f7",
    },
]

## Step 1: Query NO₂ trends via Jiskta

We use `aggregate="trend"` which runs OLS linear regression on the monthly mean NO₂ values for each grid cell in the bounding box. The result is a `slope` (µg/m³ per year) per grid cell.

We then take the **spatial mean of slopes** across all grid cells in the facility bbox. A slope of `-1.5` means the facility area is getting 1.5 µg/m³ cleaner per year.

In [ ]:
print("Querying NO₂ trends for each facility (2018–2023)...")

for c in companies:
    lat_min, lat_max = c["lat"]
    lon_min, lon_max = c["lon"]

    df_trend = client.query(
        lat=(lat_min, lat_max),
        lon=(lon_min, lon_max),
        start="2018-01",
        end="2023-12",
        variables=["no2"],
        aggregate="trend",
    )

    # aggregate=trend returns one row per grid cell with columns:
    # lat, lon, slope (µg/m³/yr), intercept, r2, n
    c["no2_slope"] = df_trend["slope"].mean()       # spatial mean of slopes
    c["no2_r2"]   = df_trend["r2"].mean()           # avg goodness of fit
    c["n_cells"]  = len(df_trend)

    print(f"  {c['name']:<35} slope={c['no2_slope']:+.2f} µg/m³/yr  r²={c['no2_r2']:.2f}  ({c['n_cells']} grid cells)")

print("\nDone.")

## Step 2: Fetch 5-year stock returns via yfinance

In [ ]:
START_DATE = "2018-01-01"
END_DATE   = "2024-01-01"

print(f"Fetching stock returns {START_DATE} → {END_DATE}...\n")

for c in companies:
    ticker = yf.Ticker(c["ticker"])
    hist = ticker.history(start=START_DATE, end=END_DATE, auto_adjust=True)

    if len(hist) < 50:
        print(f"  {c['ticker']:<12} ⚠ insufficient data")
        c["total_return_pct"] = float("nan")
        continue

    ret = (hist["Close"].iloc[-1] / hist["Close"].iloc[0] - 1) * 100
    c["total_return_pct"] = round(ret, 1)
    print(f"  {c['ticker']:<12} {ret:+.1f}%  ({len(hist)} trading days)")

## Step 3: Build the summary DataFrame

In [ ]:
df = pd.DataFrame(companies)[[
    "name", "ticker", "sector",
    "no2_slope", "no2_r2", "total_return_pct"
]]

# improvement_rate = positive number (absolute slope)
df["improvement_rate"] = -df["no2_slope"]
df = df.sort_values("improvement_rate", ascending=False).reset_index(drop=True)

print(df[[
    "name", "sector",
    "no2_slope", "no2_r2",
    "total_return_pct"
]].to_string(index=False))

## Step 4: Visualise

### Chart A — Who is actually reducing emissions?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

colors = [c["color"] for c in companies]
bars = ax.barh(
    df["name"],
    df["improvement_rate"],
    color=[next(c["color"] for c in companies if c["name"] == row["name"]) for _, row in df.iterrows()],
    height=0.55,
    edgecolor="white",
    linewidth=0.5,
)

for bar, val in zip(bars, df["improvement_rate"]):
    ax.text(
        bar.get_width() + 0.03, bar.get_y() + bar.get_height() / 2,
        f"{val:.2f} µg/m³/yr", va="center", fontsize=8.5
    )

ax.axvline(0, color="#94a3b8", linewidth=0.8)
ax.set_xlabel("Annual NO₂ improvement at facility (µg/m³ per year)")
ax.set_title(
    "NO₂ Reduction Rate at Primary Production Facility — 2018 to 2023\n"
    "Source: Copernicus CAMS reanalysis via Jiskta API, aggregate=trend",
    fontsize=11
)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.savefig("chart_a_improvement_rate.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart B — The key question: does faster improvement predict better returns?

df_plot = df.dropna(subset=["total_return_pct"]).copy()

fig, ax = plt.subplots(figsize=(9, 6))

for _, row in df_plot.iterrows():
    color = next(c["color"] for c in companies if c["name"] == row["name"])
    ax.scatter(row["improvement_rate"], row["total_return_pct"],
               color=color, s=160, zorder=5, edgecolors="white", linewidth=1)
    # nudge label to avoid overlap
    offset = (8, 6) if row["ticker"] != "TKA.DE" else (8, -14)
    ax.annotate(
        row["name"].split(" (")[0],
        (row["improvement_rate"], row["total_return_pct"]),
        textcoords="offset points", xytext=offset, fontsize=8.5,
    )

# Trend line — exclude Thyssenkrupp (restructuring outlier; see discussion)
df_fit = df_plot[df_plot["ticker"] != "TKA.DE"]
m, b = np.polyfit(df_fit["improvement_rate"], df_fit["total_return_pct"], 1)
x_line = np.linspace(df_plot["improvement_rate"].min() - 0.1,
                     df_plot["improvement_rate"].max() + 0.1, 100)
ax.plot(x_line, m * x_line + b, color="#94a3b8", linewidth=1.5,
        linestyle="--", alpha=0.8, label=f"OLS fit (excl. TKA): slope={m:.0f}%/(µg/m³/yr)")

r2 = np.corrcoef(df_fit["improvement_rate"], df_fit["total_return_pct"])[0, 1] ** 2
ax.text(0.05, 0.92, f"R² = {r2:.2f} (excl. TKA)",
        transform=ax.transAxes, fontsize=9, color="#64748b")

ax.axhline(0, color="#cbd5e1", linewidth=0.8, linestyle="-")
ax.set_xlabel("NO₂ improvement rate at facility (µg/m³ per year)\n← Less improvement     More improvement →",
              fontsize=10)
ax.set_ylabel("Total stock return, Jan 2018 → Jan 2024 (%)", fontsize=10)
ax.set_title(
    "Do Companies That Are Actually Getting Cleaner Outperform?\n"
    "6 European heavy-industry companies, Jiskta API + yfinance",
    fontsize=11, fontweight="bold"
)
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.savefig("chart_b_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nOLS slope: {m:.1f}% return per µg/m³/yr improvement")
print(f"R² (excl. TKA): {r2:.2f}")
print(f"\nThyssenkrupp note: fastest emission reducer (-{df_plot[df_plot.ticker=='TKA.DE'].improvement_rate.values[0]:.2f} µg/m³/yr)")
print(f"but market return: {df_plot[df_plot.ticker=='TKA.DE'].total_return_pct.values[0]:+.0f}% (restructuring discount)")
print("Hypothesis: decarbonisation capex is priced in with a 2-3 year lag.")

## Discussion

### What this shows

1. **RWE** is the strongest improver (-2.03 µg/m³/yr) and a strong performer (+71%). The coal-to-renewables transition is visible in satellite data and priced into the stock.

2. **BASF** barely improved (-0.46 µg/m³/yr) and is the worst-performing stock (-38%). The gas price crisis hurt, but the weak emission trajectory suggests the underlying energy mix hadn't changed — making the company more exposed when prices spiked.

3. **Thyssenkrupp** is the most interesting outlier: the *fastest* emission reducer in the dataset but the *worst* stock. The satellite data reflects genuine capex (electric arc furnace pilots, DRI projects). The market penalised the restructuring costs. If the satellite signal is right, TKA is potentially mispriced.

4. **HeidelbergMaterials** tells the opposite story: moderate improvement but strong returns (+84%) — the market is pricing in the green cement narrative (CCUS, low-carbon products) ahead of the emission signal fully appearing.

### What traditional ESG ratings miss

Standard ESG frameworks score companies on **current emission levels**. Under this approach:
- Vestas (wind turbines) scores high because its facilities are inherently clean
- ArcelorMittal scores low because steelmaking is inherently dirty

But the actionable signal is the **rate of change**: which heavy industrial operators are making the capex bets that reduce facility emissions? Satellite data captures this in near real-time. Scope 1 disclosures follow 12–18 months later.

### Extensions

- Add more companies (the API handles hundreds of bbox queries cheaply)
- Use `aggregate="monthly"` and correlate NO₂ momentum with earnings surprise windows
- Extend to Asian steel mills / Chinese power sector using CAMS Global (0.75°, 2020–present)
- Layer in ERA5 wind speed to control for meteorological dispersion effects on NO₂